In [1]:
import io
import numpy as np
import pandas as pd
import scipy.stats as stats
import plotly.express as px
import plotly.graph_objects as objects
from plotly.subplots import make_subplots
from google.colab import files
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler, OrdinalEncoder, OneHotEncoder

class PlottingMethods:
    """
    Utility class to handle modular HTML chart generation.
    """
    @staticmethod
    def generate_bar_chart(df, column, title=None):
        if df is None or column not in df.columns:
            return "<p>Error: Invalid DataFrame or Column.</p>"
        counts = df[column].value_counts().reset_index()
        counts.columns = [column, 'count']
        counts['percentage'] = (counts['count'] / counts['count'].sum() * 100).round(2)
        fig = px.bar(
            counts, x=column, y='count',
            text=counts.apply(lambda r: f"{r['count']} ({r['percentage']}%)", axis=1),
            title=title or f"Frequency Distribution of {column}"
        )
        fig.update_traces(textposition='outside')
        return fig.to_html(include_plotlyjs='cdn')

    @staticmethod
    def generate_pie_chart(df, column, title=None):
        if df is None or column not in df.columns:
            return "<p>Error: Invalid DataFrame or Column.</p>"
        fig = px.pie(df, names=column, title=title or f"Pie Chart of {column}")
        return fig.to_html(include_plotlyjs='cdn')

    @staticmethod
    def generate_histogram(df, column, title=None):
        if df is None or column not in df.columns:
            return "<p>Error: Invalid DataFrame or Column.</p>"
        fig = px.histogram(df, x=column, title=title or f"Histogram of {column}", marginal="rug")
        return fig.to_html(include_plotlyjs='cdn')


class DataInspector:
    """
    An advanced end-to-end tool for CSV data ingestion, automatic cleaning,
    feature normalization, and high-level statistical association visualization.
    """
    def __init__(self):
        self.df = None
        self.numeric_cols = []
        self.categorical_cols = []

    def _update_column_types(self):
        if self.df is not None:
            self.numeric_cols = self.df.select_dtypes(include=[np.number]).columns.tolist()
            self.categorical_cols = self.df.select_dtypes(exclude=[np.number]).columns.tolist()

    def upload_data(self):
        print("Please upload your target CSV file:")
        uploaded = files.upload()
        if not uploaded:
            print("No file uploaded.")
            return None
        file_name = list(uploaded.keys())[0]
        garbage_strings = ['?', 'n/a', 'N/A', 'NULL', 'null', ' ', '']
        self.df = pd.read_csv(io.BytesIO(uploaded[file_name]), na_values=garbage_strings)
        print(f"\nSuccessfully loaded {file_name}.")

        for col in self.df.columns:
            if self.df[col].dtype == 'object':
                try:
                    converted = pd.to_numeric(self.df[col])
                    if not converted.isna().all():
                        self.df[col] = converted
                except (ValueError, TypeError):
                    pass
        self._update_column_types()
        self.display_summary()
        return self.df

    def display_summary(self):
        if self.df is None:
            print("No active dataset. Load data using upload_data() first.")
            return
        print("\n" + "="*50)
        print("STRUCTURAL ANALYSIS SUMMARY")
        print("="*50)
        print(f"Total Rows: {self.df.shape[0]}")
        print(f"Total Columns: {self.df.shape[1]}")
        print(f"Numerical Columns ({len(self.numeric_cols)}): {self.numeric_cols}")
        print(f"Categorical Columns ({len(self.categorical_cols)}): {self.categorical_cols}")
        print("\nMissing Values Per Column:")
        print(self.df.isna().sum())
        print("\nFirst 20 Rows Preview:")
        display(self.df.head(20))

    def handle_missing_values(self, strategy='median', constant_value=None):
        if self.df is None: return
        for col in self.df.columns:
            if self.df[col].isna().sum() == 0:
                continue
            if strategy == 'constant':
                fill_val = constant_value
            elif self.df[col].dtype in [np.number]:
                if strategy == 'mean':
                    fill_val = self.df[col].mean()
                elif strategy == 'median':
                    fill_val = self.df[col].median()
                elif strategy == 'mode':
                    fill_val = self.df[col].mode()[0] if not self.df[col].mode().empty else 0
            else:
                fill_val = self.df[col].mode()[0] if not self.df[col].mode().empty else "Unknown"
            self.df[col] = self.df[col].fillna(fill_val)
        self._update_column_types()
        print(f"Missing values imputed using '{strategy}' strategy.")

    def remove_duplicates(self):
        if self.df is None: return
        initial_rows = self.df.shape[0]
        self.df.drop_duplicates(inplace=True)
        print(f"Removed {initial_rows - self.df.shape[0]} duplicate rows.")

    def handle_outliers(self, columns, action='flag'):
        if self.df is None: return
        mask = pd.Series(False, index=self.df.index)
        for col in columns:
            if col in self.numeric_cols:
                Q1 = self.df[col].quantile(0.25)
                Q3 = self.df[col].quantile(0.75)
                IQR = Q3 - Q1
                lower_bound = Q1 - 1.5 * IQR
                upper_bound = Q3 + 1.5 * IQR
                mask = mask | (self.df[col] < lower_bound) | (self.df[col] > upper_bound)
        if action == 'delete':
            initial_rows = self.df.shape[0]
            self.df = self.df[~mask]
            print(f"Deleted {initial_rows - self.df.shape[0]} outlier rows based on columns {columns}.")
        else:
            self.df['is_outlier'] = mask.astype(int)
            print(f"Outliers flagged in a new 'is_outlier' binary column.")
        self._update_column_types()

    def delete_rows(self):
        if self.df is None: return
        user_input = input("Enter comma-separated row indices to delete (e.g., 0, 5, 12): ")
        if user_input.strip():
            try:
                indices = [int(i.strip()) for i in user_input.split(',')]
                self.df.drop(index=indices, inplace=True, errors='ignore')
                print(f"Successfully pruned selected row records.")
            except ValueError:
                print("Invalid input sequence.")

    def delete_columns(self):
        if self.df is None: return
        user_input = input("Enter comma-separated exact column names to drop: ")
        if user_input.strip():
            cols = [c.strip() for c in user_input.split(',')]
            self.df.drop(columns=cols, inplace=True, errors='ignore')
            print(f"Successfully deleted columns: {cols}")
            self._update_column_types()

    def extract_normalized_numeric_data(self, strategy='standard'):
        if self.df is None or not self.numeric_cols: return pd.DataFrame()
        scaler_map = {'minmax': MinMaxScaler(), 'standard': StandardScaler(), 'robust': RobustScaler()}
        scaler = scaler_map.get(strategy, StandardScaler())
        scaled_array = scaler.fit_transform(self.df[self.numeric_cols])
        return pd.DataFrame(scaled_array, columns=[f"{c}_scaled" for c in self.numeric_cols], index=self.df.index)

    def extract_normalized_categorical_data(self, strategy='onehot'):
        if self.df is None or not self.categorical_cols: return pd.DataFrame()
        if strategy == 'onehot':
            encoder = OneHotEncoder(sparse_output=False, drop='first')
            encoded = encoder.fit_transform(self.df[self.categorical_cols])
            cols = encoder.get_feature_names_out(self.categorical_cols)
            return pd.DataFrame(encoded, columns=cols, index=self.df.index)
        elif strategy in ['ordinal', 'uniform']:
            encoder = OrdinalEncoder()
            encoded = encoder.fit_transform(self.df[self.categorical_cols])
            df_ord = pd.DataFrame(encoded, columns=[f"{c}_encoded" for c in self.categorical_cols], index=self.df.index)
            if strategy == 'uniform':
                for col in df_ord.columns:
                    if df_ord[col].max() != df_ord[col].min():
                        df_ord[col] = (df_ord[col] - df_ord[col].min()) / (df_ord[col].max() - df_ord[col].min())
            return df_ord
        return pd.DataFrame()

    def merge_engineered_dataset(self, num_strategy='standard', cat_strategy='onehot'):
        num_df = self.extract_normalized_numeric_data(strategy=num_strategy)
        cat_df = self.extract_normalized_categorical_data(strategy=cat_strategy)
        return pd.concat([self.df.copy(), num_df, cat_df], axis=1)

    def plot_univariate_subplots(self, column):
        if self.df is None or column not in self.numeric_cols:
            print("Invalid target numeric feature.")
            return
        fig = make_subplots(rows=3, cols=1, subplot_titles=(f"Violin Plot", f"Index vs Value Scatter", f"Histogram"))
        fig.add_trace(px.violin(self.df, x=column, box=True, points="all").data[0], row=1, col=1)
        fig.add_trace(px.scatter(self.df, x=self.df.index, y=column).data[0], row=2, col=1)
        fig.add_trace(px.histogram(self.df, x=column).data[0], row=3, col=1)
        fig.update_layout(height=800, title_text=f"Univariate Analysis Dashboard: {column}", showlegend=False)
        fig.show()

    def plot_relationship(self, col1, col2):
        if self.df is None or col1 not in self.df.columns or col2 not in self.df.columns:
            print("Columns not found within dataframe boundaries.")
            return
        c1_num, c2_num = col1 in self.numeric_cols, col2 in self.numeric_cols
        if c1_num and c2_num:
            fig = px.scatter(self.df, x=col1, y=col2, trendline="ols", title=f"Scatter Map: {col1} vs {col2}")
        elif not c1_num and not c2_num:
            counts = self.df.groupby([col1, col2]).size().reset_index(name='counts')
            fig = px.bar(counts, x=col1, y='counts', color=col2, barmode='group', title=f"Grouped Bar: {col1} vs {col2}")
        else:
            cat, num = (col1, col2) if not c1_num else (col2, col1)
            fig = px.box(self.df, x=cat, y=num, points="all", title=f"Distribution: {num} by {cat}")
        fig.show()

    def plot_all_associations_heatmap(self):
        if self.df is None: return
        cols = self.df.columns.tolist()
        n = len(cols)
        corr_matrix = pd.DataFrame(np.zeros((n, n)), index=cols, columns=cols)

        for i in range(n):
            for j in range(n):
                c1, c2 = cols[i], cols[j]
                if i == j:
                    corr_matrix.iloc[i, j] = 1.0
                    continue
                c1_num, c2_num = c1 in self.numeric_cols, c2 in self.numeric_cols

                if c1_num and c2_num:
                    valid_df = self.df[[c1, c2]].dropna()
                    if len(valid_df) > 1:
                        r, _ = stats.search = stats.pearsonr(valid_df[c1], valid_df[c2])
                        corr_matrix.iloc[i, j] = np.nan_to_num(r)
                elif not c1_num and not c2_num:
                    confusion_matrix = pd.crosstab(self.df[c1], self.df[c2])
                    if not confusion_matrix.empty and confusion_matrix.sum().sum() > 0:
                        chi2 = stats.chi2_contingency(confusion_matrix)[0]
                        total = confusion_matrix.sum().sum()
                        r_dim, c_dim = confusion_matrix.shape
                        if total > 0 and min(r_dim-1, c_dim-1) > 0:
                            v = np.sqrt(chi2 / (total * min(r_dim-1, c_dim-1)))
                            corr_matrix.iloc[i, j] = np.nan_to_num(v)
                else:
                    num_col, cat_col = (c1, c2) if c1_num else (c2, c1)
                    valid_df = self.df[[num_col, cat_col]].dropna()
                    groups = [group[num_col].values for name, group in valid_df.groupby(cat_col)]
                    if len(groups) > 1 and sum(len(g) for g in groups) > len(groups):
                        total_ss = ((valid_df[num_col] - valid_df[num_col].mean())**2).sum()
                        if total_ss > 0:
                            group_means = valid_df.groupby(cat_col)[num_col].mean()
                            # FIXED: This correctly uses 'group' to map standard metrics without name error
                            between_ss = sum(len(group) * (group_means[name] - valid_df[num_col].mean())**2 for name, group in valid_df.groupby(cat_col))
                            eta = np.sqrt(between_ss / total_ss)
                            corr_matrix.iloc[i, j] = np.nan_to_num(eta)

        fig = px.imshow(
            corr_matrix, text_auto=".2f",
            title="Unified Association Heatmap (Pearson r / Cramér's V / Correlation Ratio Eta)",
            color_continuous_scale="RdBu", zmin=-1.0, zmax=1.0
        )
        fig.show()

In [2]:
# Create our engine runner
inspector = DataInspector()

# Automated clean loading benchmark dataset directly into workspace
import pandas as pd
titanic_url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
inspector.df = pd.read_csv(titanic_url, na_values=['?', 'n/a', 'N/A', 'NULL', 'null', ' ', ''])
inspector._update_column_types()

# Execute core preprocessing assignment guidelines
inspector.handle_missing_values(strategy='median')
inspector.remove_duplicates()
inspector.handle_outliers(columns=['Fare'], action='delete')

Missing values imputed using 'median' strategy.
Removed 0 duplicate rows.
Deleted 116 outlier rows based on columns ['Fare'].


/tmp/ipykernel_17466/1482897931.py:106: DeprecationWarning: Converting `np.inexact` or `np.floating` to a dtype is deprecated. The current result is `float64` which is not strictly correct.
  elif self.df[col].dtype in [np.number]:


In [3]:
# 1. 3-Panel Univariate analysis for Age
inspector.plot_univariate_subplots(column='Age')

# 2. Relationship analysis Pclass vs Age (Categorical vs Numerical)
inspector.plot_relationship(col1='Pclass', col2='Age')

# 3. Relationship analysis Age vs Fare (Numerical vs Numerical)
inspector.plot_relationship(col1='Age', col2='Fare')

# 4. Generate the Fixed Unified Association Heatmap Matrix
inspector.plot_all_associations_heatmap()